# Sprint 11 · Webinar 36 · BI (Power BI + Tableau) · Solución al Proyecto Inmobiliario (100 min)
**Tema:** Estrategia comercial de Andes Capital Real Estate — construcción completa del dashboard

> En esta sesión resolverás paso a paso el **Proyecto 10** usando Power BI y Tableau:
> generarás los datasets, construirás el modelo estrella, crearás medidas analíticas
> y diseñarás tres páginas de dashboard para análisis ejecutivo, comercial y de cohortes.

**Programa:** Data Analytics · **Sprint:** 11 · **Duración:** 100 min · **Modalidad:** Práctico

> ⚠️ **Importante:** ejecuta primero la celda de Setup para generar los CSV. Todo el trabajo
> en Power BI y Tableau se realiza **fuera de este notebook**; las celdas de código
> aquí solo generan los datos y documentan los pasos.

## Fecha
> Completa aquí la fecha de la sesión.


## Objetivos de la sesión

Al finalizar esta clase, podrás:

1. Generar los **cuatro archivos CSV** del proyecto inmobiliario desde Python (incluida la tabla calendario).
2. Conectar múltiples fuentes y construir el **modelo estrella** en Power BI y Tableau.
3. Cargar `dim_fecha` como CSV y **marcarla como tabla de fechas** en Power BI.
4. Escribir **medidas DAX** (KPIs, CALCULATE, inteligencia de tiempo YoY/YTD).
5. Diseñar las **tres páginas del dashboard** (Overview, Análisis Comercial, Cohortes).
6. Replicar las métricas clave en Tableau usando **Calculated Fields y expresiones LOD**.
7. Redactar un **resumen ejecutivo** con hallazgos y recomendaciones.

## Agenda sugerida (100 minutos)

| Tiempo | Bloque | Herramienta | Contenido |
|-------:|--------|:-----------:|-----------|
| 0–10 | Setup | 🐍 Python | Generar los 4 CSV del proyecto (incluye `dim_fecha`) |
| 10–20 | Paso 1 | Power BI + Tableau | Conectar datos y modelo estrella |
| 20–30 | Paso 2 | Power BI | Cargar `dim_fecha` CSV y marcarla como tabla de fechas |
| 30–50 | Paso 3 | Power BI | Medidas: KPIs, CALCULATE, YoY/YTD |
| 50–70 | Paso 4 | Power BI | Diseño de las 3 páginas del dashboard |
| 70–85 | Paso 5–6 | Tableau | Modelo + Calculated Fields + LOD |
| 85–95 | Paso 7 | Tableau | Dashboards equivalentes |
| 95–100 | Cierre | — | Resumen ejecutivo y entrega |

---

# Setup · Generación de los datasets del proyecto (10 min) 🐍

**Meta:** producir los **4 archivos CSV** que usarás durante toda la sesión en Power BI y Tableau.

## ¿Por qué generar los datos aquí?

El proyecto trabaja con datos reales de una empresa inmobiliaria ficticia.
Generamos los datos con Python para garantizar consistencia y reproducibilidad.

> 💡 **Incluimos `dim_fecha` como CSV generado desde Python** para que el estudiante
> solo tenga que cargarlo — sin necesidad de escribir fórmulas DAX para la tabla calendario.

| Archivo | Rol en el modelo | Filas |
|---------|-----------------|------:|
| `hecho_ventas_propiedades.csv` | Tabla de hechos (transacciones) | 2 000 |
| `dim_clientes.csv` | Dimensión: quién compra | 500 |
| `dim_propiedades.csv` | Dimensión: qué se vende | 800 |
| `dim_fecha.csv` | Dimensión de tiempo (tabla calendario) | 1 096 |

> 📁 Los archivos se guardarán en la carpeta `dataset_andes_capital/`

### Imports y configuración

In [1]:
# ============================================================
# Imports y configuración — ejecuta esta celda primero
# ============================================================

import numpy as np
import pandas as pd
import os

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

OUTPUT_DIR = 'dataset_andes_capital'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('✅ Librerías cargadas.')
print('📁 Carpeta de salida:', OUTPUT_DIR)

✅ Librerías cargadas.
📁 Carpeta de salida: dataset_andes_capital


### Generador de datasets — Proyecto Inmobiliario Andes Capital (4 CSV)


In [2]:
# ============================================================
# GENERADOR DE DATASETS — Proyecto Inmobiliario Andes Capital
# Ejecuta este bloque completo para obtener los 4 CSV
# ============================================================

rng = np.random.default_rng(RANDOM_SEED)

# ── CIUDADES Y BARRIOS ──────────────────────────────────────────────────────
ciudades_barrios = {
    'Bogotá':      ['Chapinero', 'Usaquén', 'Suba', 'Kennedy', 'Teusaquillo'],
    'Medellín':    ['El Poblado', 'Laureles', 'Belén', 'Envigado', 'Sabaneta'],
    'Cali':        ['Granada', 'El Peñón', 'Ciudad Jardín', 'Chipichape', 'Centenario'],
    'Barranquilla':['El Prado', 'Manga', 'Bello Horizonte', 'Riomar', 'Alto Prado'],
    'Cartagena':   ['Bocagrande', 'Manga', 'El Cabrero', 'Crespo', 'Castillogrande'],
    'Bucaramanga': ['Cabecera', 'La Ciudadela', 'El Recreo', 'Provenza', 'Lagos'],
}
ciudades = list(ciudades_barrios.keys())

# ── 1. DIM_CLIENTES ─────────────────────────────────────────────────────────
N_CLIENTES = 500
segmentos   = ['First-time', 'Repeat', 'Investor', 'Corporate']
paises      = ['Colombia', 'México', 'Venezuela', 'Ecuador', 'Perú', 'EEUU']

ids_cliente = [f'CUST{str(i).zfill(5)}' for i in range(1, N_CLIENTES + 1)]
ciudad_cli  = rng.choice(ciudades, size=N_CLIENTES)

dim_clientes = pd.DataFrame({
    'id_cliente':         ids_cliente,
    'segmento_comprador': rng.choice(segmentos, size=N_CLIENTES,
                                     p=[0.40, 0.30, 0.20, 0.10]),
    'pais':               rng.choice(paises, size=N_CLIENTES,
                                     p=[0.70, 0.08, 0.07, 0.06, 0.05, 0.04]),
    'ciudad':             ciudad_cli,
})
dim_clientes.to_csv(f'{OUTPUT_DIR}/dim_clientes.csv', index=False)
print(f'✅ dim_clientes.csv — {len(dim_clientes)} filas')

# ── 2. DIM_PROPIEDADES ──────────────────────────────────────────────────────
N_PROPS     = 800
tipos       = ['Apartment', 'House', 'Office', 'Commercial', 'Land']
categorias  = ['Residential', 'Commercial', 'Industrial']

ids_prop  = [f'PROP{str(i).zfill(5)}' for i in range(1, N_PROPS + 1)]
ciudad_p  = rng.choice(ciudades, size=N_PROPS)
tipo_p    = rng.choice(tipos, size=N_PROPS, p=[0.40, 0.30, 0.15, 0.10, 0.05])

precios_base = {
    'Apartment': (150_000, 400_000),
    'House':     (250_000, 800_000),
    'Office':    (200_000, 600_000),
    'Commercial':(300_000, 900_000),
    'Land':      ( 80_000, 350_000),
}
precio_pub = np.array([
    rng.integers(*precios_base[t]) for t in tipo_p
], dtype=float)

barrios = [rng.choice(ciudades_barrios[c]) for c in ciudad_p]

dim_propiedades = pd.DataFrame({
    'id_propiedad':      ids_prop,
    'tipo_propiedad':    tipo_p,
    'ciudad':            ciudad_p,
    'barrio':            barrios,
    'habitaciones':      rng.integers(1, 6, size=N_PROPS),
    'tamano_m2':         rng.integers(40, 300, size=N_PROPS),
    'precio_publicado':  precio_pub.round(0),
    'categoria_propiedad': rng.choice(categorias, size=N_PROPS,
                                      p=[0.60, 0.30, 0.10]),
})
dim_propiedades.to_csv(f'{OUTPUT_DIR}/dim_propiedades.csv', index=False)
print(f'✅ dim_propiedades.csv — {len(dim_propiedades)} filas')

# ── 3. HECHO_VENTAS_PROPIEDADES ─────────────────────────────────────────────
N_VENTAS = 2_000
canales  = ['Broker', 'Digital', 'Referral', 'Direct']

# Fechas: 2022-01-01 → 2024-12-31  (3 años completos)
fecha_inicio = pd.Timestamp('2022-01-01')
fecha_fin    = pd.Timestamp('2024-12-31')
dias_total   = (fecha_fin - fecha_inicio).days

fechas_raw = pd.to_datetime(
    rng.integers(0, dias_total, size=N_VENTAS), unit='D', origin=fecha_inicio
).normalize()

idx_prop = rng.choice(N_PROPS, size=N_VENTAS, replace=True)
prop_ids  = [ids_prop[i] for i in idx_prop]
tipos_v   = [dim_propiedades.loc[i, 'tipo_propiedad'] for i in idx_prop]

descuento    = rng.uniform(0.80, 0.95, size=N_VENTAS)
precio_venta = (precio_pub[idx_prop] * descuento).round(0)

comision_por_canal = {'Broker': 0.05, 'Digital': 0.03, 'Referral': 0.04, 'Direct': 0.025}
canal_v = rng.choice(canales, size=N_VENTAS, p=[0.40, 0.30, 0.20, 0.10])
porc_com = np.array([comision_por_canal[c] + rng.uniform(-0.005, 0.005) for c in canal_v])
monto_com = (precio_venta * porc_com).round(0)

hecho = pd.DataFrame({
    'id_venta':            [f'SALE{str(i).zfill(6)}' for i in range(1, N_VENTAS + 1)],
    'fecha_venta':         fechas_raw.strftime('%Y-%m-%d'),
    'id_cliente':          rng.choice(ids_cliente, size=N_VENTAS),
    'id_propiedad':        prop_ids,
    'ciudad':              [dim_propiedades.loc[i, 'ciudad'] for i in idx_prop],
    'precio_venta':        precio_venta,
    'tipo_propiedad':      tipos_v,
    'canal_venta':         canal_v,
    'porcentaje_comision': porc_com.round(4),
    'monto_comision':      monto_com,
})
hecho.to_csv(f'{OUTPUT_DIR}/hecho_ventas_propiedades.csv', index=False)
print(f'✅ hecho_ventas_propiedades.csv — {len(hecho)} filas')

# ── 4. DIM_FECHA (tabla calendario) ─────────────────────────────────────────
# Se genera con pandas cubriendo exactamente el rango de fechas del hecho.
# Incluye todas las columnas necesarias para inteligencia de tiempo en Power BI y Tableau.

MESES_ES = {
    1: 'Enero', 2: 'Febrero', 3: 'Marzo',    4: 'Abril',
    5: 'Mayo',  6: 'Junio',   7: 'Julio',     8: 'Agosto',
    9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'
}
DIAS_ES = {
    0: 'Lunes', 1: 'Martes', 2: 'Miércoles', 3: 'Jueves',
    4: 'Viernes', 5: 'Sábado', 6: 'Domingo'
}

todas_fechas = pd.date_range(start=fecha_inicio, end=fecha_fin, freq='D')

dim_fecha = pd.DataFrame({'Date': todas_fechas})
dim_fecha['Año']            = dim_fecha['Date'].dt.year
dim_fecha['Mes']            = dim_fecha['Date'].dt.month.map(MESES_ES)
dim_fecha['Mes Numero']     = dim_fecha['Date'].dt.month
dim_fecha['Año-Mes']        = dim_fecha['Date'].dt.strftime('%Y-%m')
dim_fecha['Trimestre']      = 'Q' + dim_fecha['Date'].dt.quarter.astype(str)
dim_fecha['Año-Trimestre']  = dim_fecha['Date'].dt.year.astype(str) + '-Q' + dim_fecha['Date'].dt.quarter.astype(str)
dim_fecha['Dia Semana']     = dim_fecha['Date'].dt.dayofweek.map(DIAS_ES)
dim_fecha['Es Fin Semana']  = dim_fecha['Date'].dt.dayofweek >= 5
dim_fecha['Date']           = dim_fecha['Date'].dt.strftime('%Y-%m-%d')

dim_fecha.to_csv(f'{OUTPUT_DIR}/dim_fecha.csv', index=False)
print(f'✅ dim_fecha.csv — {len(dim_fecha)} filas  '
      f'({dim_fecha["Date"].min()} → {dim_fecha["Date"].max()})')

print('\n🏁 ¡Los 4 CSV están listos! Carga la carpeta dataset_andes_capital/ en Power BI o Tableau.')


✅ dim_clientes.csv — 500 filas
✅ dim_propiedades.csv — 800 filas
✅ hecho_ventas_propiedades.csv — 2000 filas
✅ dim_fecha.csv — 1096 filas  (2022-01-01 → 2024-12-31)

🏁 ¡Los 4 CSV están listos! Carga la carpeta dataset_andes_capital/ en Power BI o Tableau.


### Vista previa de los datasets

In [3]:
# Vista previa — ejecuta para verificar antes de ir a las herramientas BI
print('=== hecho_ventas_propiedades — primeras 5 filas ===')
display(hecho.head())

print('\n=== dim_clientes — primeras 5 filas ===')
display(dim_clientes.head())

print('\n=== dim_propiedades — primeras 5 filas ===')
display(dim_propiedades.head())

print('\n=== dim_fecha — primeras 5 filas ===')
display(dim_fecha.head())

print(f'\n📊 Rango de fechas: {hecho.fecha_venta.min()} → {hecho.fecha_venta.max()}')
print(f'📅 dim_fecha cubre: {dim_fecha["Date"].min()} → {dim_fecha["Date"].max()} ({len(dim_fecha)} días)')
print(f'💰 Ingreso total: ${hecho.precio_venta.sum():,.0f}')
print(f'🏠 Tipos de propiedad: {sorted(hecho.tipo_propiedad.unique())}')
print(f'📢 Canales de venta: {sorted(hecho.canal_venta.unique())}')


=== hecho_ventas_propiedades — primeras 5 filas ===


,id_venta,fecha_venta,id_cliente,id_propiedad,ciudad,precio_venta,tipo_propiedad,canal_venta,porcentaje_comision,monto_comision
0,SALE000001,2022-02-08,CUST00348,PROP00545,Medellín,368761.0,House,Broker,0.0495,18263.0
1,SALE000002,2022-08-09,CUST00356,PROP00796,Cartagena,244126.0,House,Broker,0.0492,12012.0
2,SALE000003,2023-02-10,CUST00087,PROP00678,Medellín,347815.0,Apartment,Broker,0.0511,17762.0
3,SALE000004,2024-03-11,CUST00486,PROP00691,Bogotá,146769.0,Apartment,Digital,0.0322,4733.0
4,SALE000005,2022-03-23,CUST00496,PROP00081,Cali,690200.0,Commercial,Digital,0.0340,23439.0



=== dim_clientes — primeras 5 filas ===


,id_cliente,segmento_comprador,pais,ciudad
0,CUST00001,Repeat,Colombia,Bogotá
1,CUST00002,Investor,Colombia,Cartagena
2,CUST00003,Repeat,Colombia,Barranquilla
3,CUST00004,First-time,Colombia,Cali
4,CUST00005,First-time,Colombia,Cali



=== dim_propiedades — primeras 5 filas ===


,id_propiedad,tipo_propiedad,ciudad,barrio,habitaciones,tamano_m2,precio_publicado,categoria_propiedad
0,PROP00001,Apartment,Cartagena,Crespo,1,238,236319.0,Industrial
1,PROP00002,Apartment,Bogotá,Usaquén,5,117,314333.0,Commercial
2,PROP00003,Apartment,Cartagena,Manga,3,285,305357.0,Residential
3,PROP00004,Apartment,Bogotá,Suba,3,94,330054.0,Residential
4,PROP00005,Commercial,Bogotá,Usaquén,3,276,413544.0,Commercial



=== dim_fecha — primeras 5 filas ===


,Date,Año,Mes,Mes Numero,Año-Mes,Trimestre,Año-Trimestre,Dia Semana,Es Fin Semana
0,2022-01-01,2022,Enero,1,2022-01,Q1,2022-Q1,Sábado,True
1,2022-01-02,2022,Enero,1,2022-01,Q1,2022-Q1,Domingo,True
2,2022-01-03,2022,Enero,1,2022-01,Q1,2022-Q1,Lunes,False
3,2022-01-04,2022,Enero,1,2022-01,Q1,2022-Q1,Martes,False
4,2022-01-05,2022,Enero,1,2022-01,Q1,2022-Q1,Miércoles,False



📊 Rango de fechas: 2022-01-01 → 2024-12-30
📅 dim_fecha cubre: 2022-01-01 → 2024-12-31 (1096 días)
💰 Ingreso total: $704,112,074
🏠 Tipos de propiedad: ['Apartment', 'Commercial', 'House', 'Land', 'Office']
📢 Canales de venta: ['Broker', 'Digital', 'Direct', 'Referral']


---

# Paso 1 · Conectar datos y construir el modelo estrella (10 min)

**Meta:** importar los 3 CSV en Power BI y Tableau, verificar tipos de datos y configurar las relaciones del esquema estrella.

---

## 1.1 · Fundamentos del esquema estrella

| Tabla | Rol | Clave |
|-------|-----|-------|
| `hecho_ventas_propiedades` | **Tabla de hechos** — registra cada transacción | PK: `id_venta` |
| `dim_clientes` | **Dimensión** — quién compra | PK: `id_cliente` |
| `dim_propiedades` | **Dimensión** — qué se vende | PK: `id_propiedad` |
| `dim_fecha` | **Dimensión de tiempo** — generada en el Setup | PK: `Date` |

**Cardinalidad:** todas las relaciones son **1 → \*** (uno en la dimensión, muchos en el hecho).  
**Dirección de filtro:** siempre **simple** (de dimensión hacia hecho).

---


## 1.2 · Paso a paso en Power BI

### Importar los 4 CSV
1. Abre **Power BI Desktop**.
2. Cinta `Inicio` → `Obtener datos` → `Texto/CSV`.
3. Navega a `dataset_andes_capital/` y carga `hecho_ventas_propiedades.csv` → **Cargar**.
4. Repite para `dim_clientes.csv`, `dim_propiedades.csv` y **`dim_fecha.csv`**.
5. Verifica en el panel **Datos** (derecha) que aparecen las **4 tablas**.

### Verificar tipos de datos en Power Query
1. Cinta `Inicio` → `Transformar datos` (Power Query).
2. En `hecho_ventas_propiedades`:
   - `fecha_venta` → **Fecha** (Date).
   - `precio_venta`, `monto_comision` → **Decimal number**.
   - `porcentaje_comision` → **Decimal number** (luego se formatea como %).
3. En `dim_fecha`:
   - `Date` → **Fecha** (Date).
   - `Año`, `Mes Numero` → **Número entero**.
   - `Es Fin Semana` → **Verdadero/Falso**.
4. Cierra y aplica (`Cerrar y aplicar`).

### Crear las relaciones (Vista Modelo)
1. Haz clic en el ícono de diagrama **Vista Modelo** (barra izquierda).
2. Arrastra `dim_clientes[id_cliente]` → `hecho_ventas_propiedades[id_cliente]`.
3. Arrastra `dim_propiedades[id_propiedad]` → `hecho_ventas_propiedades[id_propiedad]`.
4. Arrastra `dim_fecha[Date]` → `hecho_ventas_propiedades[fecha_venta]`.
5. Confirma que las **4 relaciones** muestren **1 → \*** con filtro simple (flecha única).

> ✅ **Checkpoint:** el diagrama debe verse como una estrella con `hecho_ventas_propiedades` al centro y las 4 dimensiones conectadas.

---


## 1.3 · Paso a paso en Tableau

### Conectar los datos
1. Abre **Tableau Desktop** → `Conectar` → `Archivo de texto`.
2. Navega a `dataset_andes_capital/` y abre `hecho_ventas_propiedades.csv`.
3. En el **panel de datos** (izquierda), arrastra `dim_clientes.csv` al área de relaciones.
4. Tableau detectará automáticamente el campo `id_cliente` — confírmalo.
5. Repite arrastrando `dim_propiedades.csv` y confirma la relación por `id_propiedad`.

### Verificar tipos de datos
- `fecha_venta` → tipo **Date** (icono de calendario).
- `precio_venta`, `monto_comision`, `porcentaje_comision` → tipo **Number (decimal)**.

> ✅ **Checkpoint:** en el panel de relaciones deben aparecer las 3 tablas conectadas.
> Tableau usa **relaciones lógicas** — no necesita dim_fecha como tabla física.

> 💡 **Diferencia clave Power BI vs Tableau:**
> Power BI almacena en memoria (VertiPaq) y requiere relaciones explícitas.
> Tableau evalúa las relaciones en tiempo de consulta — más flexible pero más lento en volúmenes grandes.

In [4]:
# (Sin código — este paso se realiza en Power BI y Tableau)

---

# Paso 2 · Configurar `dim_fecha` como tabla de fechas en Power BI (10 min)

**Meta:** marcar el CSV `dim_fecha` ya cargado como tabla de fechas oficial para habilitar
las funciones de inteligencia de tiempo de DAX.

> ✅ **No necesitas escribir ninguna fórmula DAX** — la tabla ya fue generada en Python con todas las columnas necesarias.  
> En Tableau usarás directamente las funciones de fecha nativas sobre `fecha_venta`.

---


## 2.1 · ¿Por qué marcar dim_fecha?

Las funciones DAX de inteligencia de tiempo (`TOTALYTD`, `SAMEPERIODLASTYEAR`, etc.)
requieren que Power BI identifique **una única tabla de fechas marcada explícitamente**.
Sin este paso, las funciones de tiempo no funcionarán correctamente.

La tabla `dim_fecha.csv` ya incluye todas las columnas necesarias:

| Columna | Tipo | Ejemplo |
|---------|------|---------|
| `Date` | Fecha | `2022-01-05` |
| `Año` | Entero | `2022` |
| `Mes` | Texto | `Enero` |
| `Mes Numero` | Entero | `1` |
| `Año-Mes` | Texto | `2022-01` |
| `Trimestre` | Texto | `Q1` |
| `Año-Trimestre` | Texto | `2022-Q1` |
| `Dia Semana` | Texto | `Miércoles` |
| `Es Fin Semana` | Booleano | `FALSE` |

## 2.2 · Marcar dim_fecha como tabla de fechas

1. En la **Vista Modelo** o panel **Datos**, selecciona la tabla `dim_fecha`.
2. Cinta superior → `Herramientas de tabla` → **`Marcar como tabla de fechas`**.
3. En el diálogo, selecciona la columna **`Date`** como columna de fecha → **Aceptar**.
4. Power BI validará que no haya fechas duplicadas ni días faltantes — debe mostrar ✅.

## 2.3 · Verificar la relación con la tabla de hechos

La relación `dim_fecha[Date]` → `hecho_ventas_propiedades[fecha_venta]` ya fue creada
en el Paso 1. Confirma que:

- Cardinalidad: **1 → \*** (un día → muchas ventas).
- Dirección de filtro: **simple** (flecha apunta hacia el hecho).
- Estado: **activa** (línea continua, no punteada).

> ✅ **Checkpoint:** crea una tarjeta rápida en el canvas con `dim_fecha[Año-Mes]` y
> `COUNT(hecho_ventas_propiedades[id_venta])` — si filtra correctamente por mes, el modelo está listo.

## 2.4 · Nota para Tableau

En Tableau **no necesitas ninguna configuración adicional** de tabla de fechas.
Usa directamente `YEAR([fecha_venta])`, `MONTH([fecha_venta])`, `DATETRUNC()` en tus hojas.
Si quieres cargar `dim_fecha.csv`, puedes hacerlo como tabla de dimensión conectada por `Date = fecha_venta`,
pero **no es obligatorio**.


In [5]:
# (Sin código — este paso se realiza en Power BI Desktop con DAX)

---

# Paso 3 · Creación de medidas en Power BI (20 min)

**Meta:** construir todas las medidas DAX que responden las preguntas de negocio del dashboard.

---

## 3.1 · Medidas base (KPIs principales)

Crea una **carpeta de medidas** llamada `_Medidas` para mantener el modelo organizado:
> En el panel Datos, haz clic derecho sobre cualquier tabla → `Nueva carpeta de medidas`.

Crea cada medida con `Cinta Modelado → Nueva medida`:

```dax
-- ① Ingreso Total
Ingreso Total =
SUM(hecho_ventas_propiedades[precio_venta])

-- ② Cantidad de Ventas
Cantidad Ventas =
COUNTROWS(hecho_ventas_propiedades)

-- ③ Ticket Promedio
Ticket Promedio =
DIVIDE([Ingreso Total], [Cantidad Ventas], 0)

-- ④ Comisión Total
Comision Total =
SUM(hecho_ventas_propiedades[monto_comision])
```

> 💡 Formatea `Ingreso Total` y `Comisión Total` como **Moneda**, `Ticket Promedio` como **Moneda**,
> `Cantidad Ventas` como **Número entero** (sin decimales).

---

## 3.2 · Medidas con CALCULATE (contexto de filtro)

```dax
-- ⑤ Ingreso por Apartamentos (ejemplo de CALCULATE con filtro fijo)
Ingreso Apartamentos =
CALCULATE(
    [Ingreso Total],
    hecho_ventas_propiedades[tipo_propiedad] = "Apartment"
)

-- ⑥ Ingreso por Canal Broker
Ingreso Broker =
CALCULATE(
    [Ingreso Total],
    hecho_ventas_propiedades[canal_venta] = "Broker"
)

-- ⑦ % del Total (participación de cada fila sobre el total general)
% del Total Ingreso =
DIVIDE(
    [Ingreso Total],
    CALCULATE([Ingreso Total], ALL(hecho_ventas_propiedades)),
    0
)
```

> 💡 `ALL()` elimina todos los filtros del contexto visual, permitiendo calcular la participación relativa.

---

## 3.3 · Inteligencia de tiempo (YTD, YoY, MoM)

Estas medidas requieren que `dim_fecha` esté marcada como tabla de fechas.

```dax
-- ⑧ Ingreso YTD (acumulado del año en curso)
Ingreso YTD =
TOTALYTD(
    [Ingreso Total],
    dim_fecha[Date]
)

-- ⑨ Ingreso Año Anterior (mismo período del año pasado)
Ingreso Año Anterior =
CALCULATE(
    [Ingreso Total],
    SAMEPERIODLASTYEAR(dim_fecha[Date])
)

-- ⑩ Crecimiento YoY (Year over Year)
YoY % =
DIVIDE(
    [Ingreso Total] - [Ingreso Año Anterior],
    [Ingreso Año Anterior],
    BLANK()
)

-- ⑪ Ingreso Mes Anterior (MoM)
Ingreso Mes Anterior =
CALCULATE(
    [Ingreso Total],
    PREVIOUSMONTH(dim_fecha[Date])
)

-- ⑫ Crecimiento MoM
MoM % =
DIVIDE(
    [Ingreso Total] - [Ingreso Mes Anterior],
    [Ingreso Mes Anterior],
    BLANK()
)
```

> ✅ **Checkpoint:** crea una tabla simple en el canvas con `dim_fecha[Año-Mes]`, `Ingreso Total`,
> `Ingreso YTD` y `YoY %` para verificar que los valores son coherentes.

---

## 3.4 · Medidas para análisis de cohortes

```dax
-- ⑬ Fecha Primera Compra por cliente
Primera Compra Cliente =
CALCULATE(
    MIN(hecho_ventas_propiedades[fecha_venta]),
    ALLEXCEPT(hecho_ventas_propiedades, hecho_ventas_propiedades[id_cliente])
)

-- Columna calculada en hecho_ventas_propiedades (no medida):
-- Año-Mes Cohorte = FORMAT([Primera Compra Cliente], "YYYY-MM")
-- Crear con: Cinta Modelado → Nueva columna (sobre la tabla de hechos)

-- ⑭ Clientes Únicos
Clientes Unicos =
DISTINCTCOUNT(hecho_ventas_propiedades[id_cliente])

-- ⑮ Clientes que volvieron a comprar (más de 1 transacción)
Clientes Recurrentes =
COUNTROWS(
    FILTER(
        VALUES(hecho_ventas_propiedades[id_cliente]),
        CALCULATE(COUNTROWS(hecho_ventas_propiedades)) > 1
    )
)

-- ⑯ Tasa de Recurrencia
Tasa Recurrencia =
DIVIDE([Clientes Recurrentes], [Clientes Unicos], 0)
```

In [6]:
# (Sin código — este paso se realiza en Power BI Desktop con DAX)

---

# Paso 4 · Diseño del Dashboard en Power BI (20 min)

**Meta:** construir las 3 páginas del dashboard con los visuales requeridos por el proyecto.

---

## 4.1 · Página 1: Overview Ejecutivo

**Propósito:** mostrar el desempeño general del negocio de un vistazo.

### Configuración de la página
- Renombra la pestaña: `📊 Overview`.
- Fondo: blanco o gris muy claro. Fuente del dashboard: Segoe UI.

### Visuales requeridos

| # | Visual | Campos |
|---|--------|--------|
| 1 | **4 Tarjetas KPI** | `Ingreso Total`, `Cantidad Ventas`, `Ticket Promedio`, `Comisión Total` |
| 2 | **Gráfico de líneas** (tendencia mensual) | Eje X: `dim_fecha[Año-Mes]` · Valores: `Ingreso Total` |
| 3 | **Barra apilada** (ingresos por ciudad) | Eje: `hecho_ventas_propiedades[ciudad]` · Valores: `Ingreso Total` |
| 4 | **Tarjeta o visual KPI** (YoY) | Valor: `YoY %` · Objetivo: 0% |

### Pasos de construcción
1. Página en blanco → inserta 4 **Tarjeta** (Card): una por KPI.
2. Inserta **Gráfico de líneas**: eje X = `dim_fecha[Año-Mes]`, valor = `Ingreso Total`.
   - En la sección **Análisis** del visual, activa la **Línea de tendencia**.
3. Inserta **Gráfico de barras agrupadas**: eje = `ciudad`, valor = `Ingreso Total`.
   - Ordena de mayor a menor (botón de orden en el encabezado del visual).
4. Inserta **Segmentaciones** (Slicers): `dim_fecha[Año]`, `hecho_ventas_propiedades[tipo_propiedad]`.

> ✅ **Resultado esperado:** con el slicer de año puedes comparar 2022, 2023 y 2024 instantáneamente.

---

## 4.2 · Página 2: Análisis Comercial

**Propósito:** analizar qué productos, canales y clientes generan más ingresos.

### Visuales requeridos

| # | Visual | Campos |
|---|--------|--------|
| 1 | **Gráfico de barras** | Eje: `tipo_propiedad` · Valores: `Ingreso Total` |
| 2 | **Gráfico de barras** | Eje: `canal_venta` · Valores: `Ingreso Total` |
| 3 | **Gráfico de barras** | Eje: `segmento_comprador` (de dim_clientes) · Valores: `Cantidad Ventas` |
| 4 | **Tabla de detalle** | `tipo_propiedad`, `Ingreso Total`, `% del Total Ingreso`, `Ticket Promedio` |
| 5 | **Gráfico circular** (opcional) | `canal_venta` vs `Ingreso Total` |

### Pasos de construcción
1. Inserta **Gráfico de barras horizontales**: eje Y = `tipo_propiedad`, valor X = `Ingreso Total`.
   - Activa **Etiquetas de datos** → formato: moneda sin decimales.
2. Inserta segundo gráfico de barras para `canal_venta`.
3. Para el gráfico de `segmento_comprador`: necesitas la relación activa con `dim_clientes`.
   - Arrastra `dim_clientes[segmento_comprador]` al eje.
4. Inserta **Tabla**: columnas = `tipo_propiedad`, `Ingreso Total`, `% del Total Ingreso`, `Ticket Promedio`.
   - Aplica **formato condicional** en `% del Total Ingreso` → barra de datos.

---

## 4.3 · Página 3: Análisis de Cohortes y Recurrencia

**Propósito:** entender si los clientes vuelven a comprar y cuándo.

### Visuales requeridos

| # | Visual | Campos |
|---|--------|--------|
| 1 | **Matriz de cohortes** | Filas: `Año-Mes Cohorte` · Columnas: `dim_fecha[Año-Mes]` · Valores: `Clientes Unicos` |
| 2 | **Tarjetas KPI** | `Clientes Unicos`, `Clientes Recurrentes`, `Tasa Recurrencia` |
| 3 | **Gráfico de líneas** | Evolución de `Clientes Unicos` por mes |

### Pasos de construcción — Matriz de cohortes
1. Asegúrate de haber creado la **columna calculada** `Año-Mes Cohorte` en la tabla de hechos:
   ```dax
   Año-Mes Cohorte = FORMAT(
       CALCULATE(MIN(hecho_ventas_propiedades[fecha_venta]),
                 ALLEXCEPT(hecho_ventas_propiedades, hecho_ventas_propiedades[id_cliente])),
       "YYYY-MM"
   )
   ```
2. Inserta un visual **Matriz**:
   - **Filas:** `hecho_ventas_propiedades[Año-Mes Cohorte]`
   - **Columnas:** `dim_fecha[Año-Mes]`
   - **Valores:** `Clientes Unicos`
3. En Formato visual → **Formato condicional** sobre los valores → escala de color (blanco→azul).
4. Esta visualización funciona como un **heatmap de retención**.

> 💡 La diagonal de la matriz representa la cohorte en su mes de adquisición.
> Los valores fuera de la diagonal muestran clientes que repitieron compra.

> ✅ **Checkpoint:** publica el informe en **Power BI Service** (`Archivo → Publicar`) y copia el enlace.

In [7]:
# (Sin código — este paso se realiza en Power BI Desktop)

---

# Paso 5 · Modelo y campos calculados en Tableau (15 min)

**Meta:** replicar la lógica analítica del proyecto en Tableau usando Calculated Fields y expresiones LOD.

---

## 5.1 · Verificación del modelo (desde el Paso 1)

En la pestaña **Fuente de datos** de Tableau, confirma:

- `hecho_ventas_propiedades` en el centro (tabla de hechos).
- `dim_clientes` conectada por `id_cliente` (relación lógica).
- `dim_propiedades` conectada por `id_propiedad` (relación lógica).
- `fecha_venta` reconocida como tipo **Date**.

> 💡 En Tableau **no necesitas dim_fecha** como tabla física. Las funciones
> `YEAR()`, `MONTH()`, `DATETRUNC()` operan directamente sobre `fecha_venta`.

---

## 5.2 · Crear campos calculados básicos (KPIs)

En Tableau, los Calculated Fields equivalen a las medidas DAX.

**Cómo crear:** clic derecho en el panel Datos → `Crear campo calculado...`

```
// ① Ingreso Total
SUM([precio_venta])

// ② Cantidad de Ventas
COUNT([id_venta])

// ③ Ticket Promedio
SUM([precio_venta]) / COUNT([id_venta])

// ④ Comisión Total
SUM([monto_comision])
```

---

## 5.3 · Expresiones LOD — equivalente a CALCULATE

Las **expresiones LOD** (Level of Detail) permiten calcular a un nivel de granularidad
diferente al del visual, equivalente a `CALCULATE` con `ALL`/`ALLEXCEPT` en DAX.

| Objetivo | DAX | Tableau LOD |
|----------|-----|-------------|
| Total ignorando filtros | `CALCULATE([Ingreso Total], ALL(...))` | `{ FIXED : SUM([precio_venta]) }` |
| Total por tipo de propiedad | `CALCULATE([Ingreso], ALLEXCEPT(..., tipo))` | `{ FIXED [tipo_propiedad] : SUM([precio_venta]) }` |
| Primera compra por cliente | `MIN` con `ALLEXCEPT` | `{ FIXED [id_cliente] : MIN([fecha_venta]) }` |

### Campos LOD a crear

```
// ⑤ Total General (para % del total)
{ FIXED : SUM([precio_venta]) }
// → Renombrar: "Total General"

// ⑥ % del Total
SUM([precio_venta]) / [Total General]
// → Renombrar: "% Participación"
// → Formato: Porcentaje (2 decimales)

// ⑦ Fecha primera compra por cliente
{ FIXED [id_cliente] : MIN([fecha_venta]) }
// → Renombrar: "Primera Compra"

// ⑧ Año-Mes Cohorte
DATENAME('year', [Primera Compra]) + '-' +
RIGHT('0' + STR(DATEPART('month', [Primera Compra])), 2)
// → Renombrar: "Cohorte (Año-Mes)"
```

---

## 5.4 · Inteligencia de tiempo en Tableau

Tableau calcula YoY y MoM con funciones de fecha y filtros de contexto:

```
// ⑨ YoY % (crecimiento año sobre año)
(SUM([precio_venta]) - LOOKUP(SUM([precio_venta]), -1)) /
ABS(LOOKUP(SUM([precio_venta]), -1))
// → Usar en una vista de tabla con YEAR(fecha_venta) como dimensión
// → Activar: Análisis → Cálculo de tabla → Diferencia porcentual desde el anterior

// ⑩ Para YTD: usar filtro de fecha relativa
// En cualquier visual → clic derecho sobre la fecha → Filtro → Relativo a hoy → Año hasta la fecha
```

> 💡 En Tableau la forma más directa de obtener YTD es usar **Cálculos de tabla rápidos**:
> clic derecho sobre la medida en el visual → `Cálculo de tabla rápido` → `Total acumulado`.

> Para YoY: clic derecho → `Agregar cálculo de tabla secundario` → `Diferencia porcentual respecto a`.

---

In [8]:
# (Sin código — este paso se realiza en Tableau Desktop)

---

# Paso 6 · Construir los dashboards en Tableau (10 min)

**Meta:** crear las 3 hojas y ensamblarlas en un dashboard interactivo de Tableau.

---

## 6.1 · Hojas de trabajo requeridas

Crea una hoja (Sheet) por visual antes de armar el dashboard.

| Hoja | Tipo de visual | Dimensiones | Métricas |
|------|---------------|-------------|---------|
| `Tendencia Mensual` | Línea | `MONTH(fecha_venta)`, `YEAR(fecha_venta)` (color) | `Ingreso Total` |
| `Ingresos por Ciudad` | Barras horizontales | `ciudad` | `Ingreso Total` |
| `Tipo Propiedad` | Barras | `tipo_propiedad` | `Ingreso Total` |
| `Canal Venta` | Barras | `canal_venta` | `Ingreso Total` |
| `Segmento Cliente` | Barras | `segmento_comprador` | `COUNT(id_venta)` |
| `Mapa de Calor Cohortes` | Texto / Heatmap | `Cohorte (Año-Mes)`, `MONTH/YEAR(fecha_venta)` | `COUNTD(id_cliente)` |

## 6.2 · Ensamblar el Dashboard 1 — Overview Ejecutivo

1. Pestaña inferior → `Nueva hoja de panel`.
2. Tamaño: **1 366 × 768** (pantalla estándar) o Automático.
3. Arrastra desde el panel izquierdo:
   - `Tendencia Mensual` → zona central superior.
   - `Ingresos por Ciudad` → zona inferior izquierda.
4. Inserta **Objetos de texto** con los KPIs:
   - Crea una hoja especial: eje vacío, solo una tarjeta con el valor de `Ingreso Total`.
   - Repite para los 4 KPIs y alinéalos en la parte superior.
5. Inserta **Filtros interactivos**: arrastra `YEAR(fecha_venta)` y `tipo_propiedad` al dashboard
   → clic derecho → `Usar como filtro`.

## 6.3 · Dashboard 2 — Análisis Comercial

1. Nueva hoja de panel → arrastra `Tipo Propiedad`, `Canal Venta`, `Segmento Cliente`.
2. Añade una **Tabla de detalle**: hoja con `tipo_propiedad`, `Ingreso Total`, `% Participación`, `Ticket Promedio`.
3. Activa filtros cruzados: selecciona cualquier barra → los demás visuales se filtran automáticamente.

## 6.4 · Dashboard 3 — Cohortes

1. Nueva hoja de panel → arrastra `Mapa de Calor Cohortes`.
2. Configura la hoja `Mapa de Calor Cohortes`:
   - Filas: `Cohorte (Año-Mes)`.
   - Columnas: `DATETRUNC('month', fecha_venta)` (formato Año-Mes).
   - Marca: `Texto` con `COUNTD(id_cliente)`.
   - Color: `COUNTD(id_cliente)` → paleta Naranja-Azul divergente.
3. Añade tarjetas con `Clientes Unicos`, `Clientes Recurrentes`.

## 6.5 · Publicar en Tableau Public

1. `Archivo` → `Guardar en Tableau Public como...`
2. Inicia sesión con tu cuenta de Tableau Public.
3. Copia el enlace del dashboard publicado.

> ✅ **Checkpoint final:** ambas herramientas (Power BI y Tableau) deben mostrar valores similares
> de `Ingreso Total`, `Cantidad Ventas` y distribución por `tipo_propiedad`.

In [9]:
# (Sin código — este paso se realiza en Tableau Desktop)

---

# Cierre · Resumen Ejecutivo y Entrega (10 min)

**Meta:** interpretar los resultados del dashboard y documentar hallazgos estratégicos.

---

## Guía para redactar el resumen ejecutivo

Una vez construido el dashboard, completa el Paso 6 del notebook del proyecto.
Usa esta estructura como referencia:

### Hallazgos clave (obtén los valores reales de tu dashboard)

- El **ingreso total** del período 2022–2024 fue de **$___**, generado por **___** transacciones.
- El tipo de propiedad con mayor revenue es **___**, representando el **___**% del ingreso total.
- La ciudad con mayor volumen de ventas es **___**.
- El canal de venta más eficiente es **___** (mayor ingreso) vs **___** (mayor volumen de transacciones).
- El negocio mostró un crecimiento YoY de **___**% entre 2022 y 2023, y **___**% entre 2023 y 2024.

### Insights accionables

- Si el segmento `Investor` genera alto ticket promedio pero bajo volumen → estrategia: **captación focalizada**.
- Si `Broker` tiene alto ingreso pero alta comisión → evaluar si el canal `Digital` es más rentable.
- Si la retención de cohortes cae bruscamente después del mes 1 → implementar **programa de fidelización**.

### Recomendaciones estratégicas

1. **Concentrar esfuerzo comercial** en los tipos de propiedad y ciudades de mayor margen.
2. **Optimizar el mix de canales**: reducir dependencia del canal más costoso si hay alternativas rentables.
3. **Mejorar retención**: los datos de cohortes revelan el momento exacto donde se pierde al cliente.

---

## Instrucciones de entrega

Completa la **celda de entrega** en el notebook del proyecto con tu enlace.

```
✅ Si trabajaste en Power BI:
   - Enlace al dashboard publicado en Power BI Service, O
   - Enlace de Drive/OneDrive al archivo .pbix

✅ Si trabajaste en Tableau:
   - Enlace al dashboard publicado en Tableau Public
```

> 💡 El dashboard será evaluado con la **Grading Rubric** del proyecto, que evalúa:
> limpieza de datos, modelo estrella, medidas correctas y calidad visual del dashboard.

In [10]:
# Enlace del proyecto (pega aquí tu link)
link_proyecto = ""
print("🚀 Proyecto entregado:", link_proyecto if link_proyecto else "⚠️ Pendiente de completar")

🚀 Proyecto entregado: ⚠️ Pendiente de completar
